## Esto se uso para importar Puntos de Parada (from SHP) y Line Routes a Visum 
- Era cuando estabamos trabajando sobre la red de TransCAD 
- ya no usamos esta red y esto no se usa
- __sirve para la documentación de como crear objetos SP & LineRoute en Visum__

In [1]:
import win32com.client as com
import os
import pandas as pd
import numpy as np
from functools import lru_cache

## Open Visum File .ver

In [2]:
#Open network .ver file (from local disk not onedrive)
import win32com.client

Visum = com.Dispatch("Visum.Visum.250") #Add .250 for Visum 25 version
Visum.LoadVersion("C:/Users/AP03542515/Documents/Import RedGDL/RedGDL_withTravelTime.ver")
C = win32com.client.constants

In [3]:
#Select network
Net = Visum.Net

## Read StopPoints CSV (paradas logicas insumo de IMEPLAN)

In [4]:
sp2024_csv = r"C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Red Vial Guadalajara\Red Vial Guadalajara\Importacion TransCAD\Transporte Publico\Puntos de Parada\PuntosParada2024_Attributes.csv"
sp2024_df = pd.read_csv(sp2024_csv)
sp2024_df

,ID,ROUTE_ID,PASS_COUNT,MILEPOST,STOP_ID,USERID,TRACK,SECUENCIA,LINK_ID,DIR,NODE,TIPO
0,1,1,1,0.000000,1,0.0,1.0,1.0,120551.0,1.0,85179,Parada
1,2,1,1,0.459851,2,459.0,1.0,2.0,140696.0,1.0,52849,Parada
2,3,1,1,0.646481,3,646.0,1.0,5.0,79908.0,-1.0,52902,Parada
3,4,1,1,0.872291,4,872.0,1.0,7.0,79910.0,-1.0,52904,Parada
4,5,1,1,1.219899,5,1219.0,1.0,9.0,144971.0,1.0,53616,Parada
...,...,...,...,...,...,...,...,...,...,...,...,...
45337,46314,1337,1,9.013195,46314,21640.0,521.0,210.0,117476.0,-1.0,82302,Parada
45338,46315,1337,1,9.230524,46315,21858.0,521.0,215.0,66355.0,-1.0,44351,Parada
45339,46316,1337,1,9.652481,46316,22280.0,521.0,218.0,66358.0,-1.0,44333,Parada
45340,46317,1337,1,0.042722,46317,NaN,NaN,NaN,NaN,NaN,45351,NaN


#### __comment:__ Los registros que tienen campo SECUENCIA en NULL... NO SIRVEN, hay que quitarlos
#### puede ser solo una parada en la ruta la que se borra o puede ser que toda una ruta no tenga secuencia en sus paradas (ej.1104, 1105, etc...)

In [119]:
no_sequence = sp2024_df[sp2024_df['SECUENCIA'].isna()]
no_sequence = no_sequence.sort_values(by=['ROUTE_ID'])
no_sequence

,ID,ROUTE_ID,PASS_COUNT,MILEPOST,STOP_ID,USERID,TRACK,SECUENCIA,LINK_ID,DIR,NODE,TIPO
41396,44753,3,1,35.385487,44753,NaN,NaN,NaN,NaN,NaN,4547,Extra
41395,44754,5,1,35.384651,44754,NaN,NaN,NaN,NaN,NaN,4547,Extra
41334,44759,10,1,7.386519,44759,NaN,NaN,NaN,NaN,NaN,98295,Extra
41333,44760,11,1,28.117382,44760,NaN,NaN,NaN,NaN,NaN,98295,Extra
44531,45387,215,1,3.226703,45387,NaN,NaN,NaN,NaN,NaN,72865,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
45230,46207,1333,1,0.032434,46207,NaN,NaN,NaN,NaN,NaN,83646,NaN
45286,46263,1335,1,0.034565,46263,NaN,NaN,NaN,NaN,NaN,83646,NaN
45287,46264,1335,1,0.570972,46264,NaN,NaN,NaN,NaN,NaN,83429,NaN
45313,46290,1336,1,9.784491,46290,NaN,NaN,NaN,NaN,NaN,83653,NaN


In [ ]:
sp2024_withSequence = sp2024_df.dropna(subset=['SECUENCIA']) #1,568 registros sin SECUENCIA
len(sp2024_withSequence) #Quedaron 43,774 puntos de parada x Ruta

43774

### obtener paradas fisicas filtrando los unique NODES

In [15]:
#Unique nodes in sp2024_withSequence
unique_nodes = sp2024_withSequence["NODE"].unique()
len(unique_nodes)

7374

## Create Stop Points ON NODES

In [16]:
stop_point_no = 1

for node in unique_nodes:
    node_id = node
    
    #Add STOP
    stop = Net.AddStop(stop_point_no)
    #Add STOP AREA on NODE
    stop_area = Net.AddStopArea(stop_point_no, stop_point_no, node_id)
    #Add STOP POINT on NODE
    stop_point = Net.AddStopPointOnNode(stop_point_no, stop_point_no, node_id)

    
    #increment stop point number for next stop point (used for ids)
    stop_point_no += 1

### Read Stop Points from VISUM & join with Stop_Points CSV on NODE

In [17]:
visum_stop_points = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.StopPoints.GetMultiAttValues("No")],
    "NodeNo": [i[1] for i in Visum.Net.StopPoints.GetMultiAttValues("NodeNo")],
})
visum_stop_points = visum_stop_points.astype(int)
visum_stop_points

,No,NodeNo
0,1,85179
1,2,52849
2,3,52902
3,4,52904
4,5,53616
...,...,...
7369,7370,50803
7370,7371,51913
7371,7372,51958
7372,7373,83434


### Join SP df with StopPointNo (ID) from Visum

In [18]:
stop_points = pd.merge(sp2024_withSequence, visum_stop_points, left_on="NODE", right_on="NodeNo", how="left")
stop_points = stop_points.rename(columns={"No": "StopPointNo"})

#Remove rows with missing values in SECUENCIA or DIR columns (45,342 paradas x ruta --> 43,774 paradas x ruta)
#From original stop points df csv 1,568 rows had missing values in SECUENCIA or DIR columns, which are necessary for assigning stop points to a line route. 
# These rows will be removed 
stop_points_cleaned = stop_points.dropna(subset=["SECUENCIA", "DIR"])
stop_points_cleaned = stop_points_cleaned.astype({"SECUENCIA": int, "DIR": int})
stop_points_cleaned

,ID,ROUTE_ID,PASS_COUNT,MILEPOST,STOP_ID,USERID,TRACK,SECUENCIA,LINK_ID,DIR,NODE,TIPO,StopPointNo,NodeNo
0,1,1,1,0.000000,1,0.0,1.0,1,120551.0,1,85179,Parada,1,85179
1,2,1,1,0.459851,2,459.0,1.0,2,140696.0,1,52849,Parada,2,52849
2,3,1,1,0.646481,3,646.0,1.0,5,79908.0,-1,52902,Parada,3,52902
3,4,1,1,0.872291,4,872.0,1.0,7,79910.0,-1,52904,Parada,4,52904
4,5,1,1,1.219899,5,1219.0,1.0,9,144971.0,1,53616,Parada,5,53616
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43769,46313,1337,1,8.795644,46313,21423.0,521.0,204,66298.0,-1,1797,Parada,6378,1797
43770,46314,1337,1,9.013195,46314,21640.0,521.0,210,117476.0,-1,82302,Parada,6387,82302
43771,46315,1337,1,9.230524,46315,21858.0,521.0,215,66355.0,-1,44351,Parada,6388,44351
43772,46316,1337,1,9.652481,46316,22280.0,521.0,218,66358.0,-1,44333,Parada,6389,44333


## Read LineRoutes CSV (rutas publicas insumo de IMEPLAN)

In [89]:
#csv puntos parada 2024
ro2024_csv = r"C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Red Vial Guadalajara\Red Vial Guadalajara\Importacion TransCAD\Transporte Publico\Rutas Operando\RutasOperando2024_Attributes.csv"
ro2024_df = pd.read_csv(ro2024_csv)
ro2024_df

,ROUTE_ID,ROUTE_NAME,ID_RUTA,TIME,LONGITUD,DISTANCE,RUTA_ACT,RUTA_ANT,CLASE,ORI_DES,...,FLOTA,CAPACIDAD,HEADWAY,TDR,MODE,CORREDOR,DEMANDA1,DEMANDA2,DEMANDA3,F_CORREDOR
0,1,Route 1,1,36.05,9.55,0.0,C33 Via Corta,NaN,Complementarias,NaN,...,0.0,240.0,20.0,31.63,1,NaN,0.0,0.0,0.00,0.0000
1,2,Route 10,10,68.46,16.93,0.0,C02,45A_2,Complementarias,Destino:�Centro Metropolitano,...,17.0,536.0,9.0,60.07,1,NaN,0.0,0.0,0.00,0.0000
2,3,Route 100,100,116.25,39.28,3874.0,C125 Valle de los Emperadores,186 Valle de los Emperadores_2,Complementarias,Destino: Antigua Central de Autobuses,...,29.0,404.2,14.0,121.80,1,11.0,0.0,0.0,2.07,98.6186
3,4,Route 101,101,114.35,38.50,3904.0,C125 Valle de los Emperadores directa,186 Valle de los Emperadores - Direct_1,Complementarias,Origen: Valle de los Emperadores,...,21.0,297.0,18.0,120.11,1,11.0,0.0,0.0,3.16,101.4041
4,5,Route 102,102,116.25,39.28,3874.0,C125 Valle de los Emperadores directa,186 Valle de los Emperadores - Direct_2,Complementarias,Destino: Antigua Central de Autobuses,...,21.0,297.0,18.0,121.80,1,11.0,0.0,0.0,2.07,98.6186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
791,1333,Route 952,952,69.92,17.35,0.0,L3 Alimentadora Milpillas Op3,L3,SITREN,Milpillas,...,5.0,112.0,20.0,81.21,6,10.0,0.0,0.0,0.00,0.0000
792,1334,Route 953,953,94.06,12.54,0.0,L3 Alimentadora Angeles Op3,L3,SITREN,�ngeles,...,9.0,152.0,20.0,86.09,6,NaN,0.0,0.0,0.00,0.0000
793,1335,Route 954,954,98.18,13.31,0.0,L3 Alimentadora Angeles Op3,L3,SITREN,�ngeles,...,9.0,152.0,20.0,89.98,6,NaN,0.0,0.0,0.00,0.0000
794,1336,Route 955,955,83.77,9.64,0.0,L3 Alimentadora Las Agujas Op3,L3,SITREN,T16B-C03,...,20.0,440.0,12.0,76.86,6,NaN,0.0,0.0,0.00,0.0000


### __que haces con las rutas repetidas que se supone una es de ida y otra de regreso pero por ejemplo la 1003 y 1004 son exactamente la misma tiene los mismos NODES en la misma seq. con las mismas direcciones PERO NO PUEDES SABER QUE ES LA MISMA RUTA PORQUE TIENE DIFERENTE ROUTE_ID solo lo sabrias si ves sus elementos

In [26]:
sp2024_withSequence[sp2024_withSequence['ROUTE_ID'] == 1003]

,ID,ROUTE_ID,PASS_COUNT,MILEPOST,STOP_ID,USERID,TRACK,SECUENCIA,LINK_ID,DIR,NODE,TIPO
2853,2854,1003,1,0.190977,2854,190.0,135.0,1.0,145013.0,-1.0,105863,Parada
2854,2855,1003,1,0.376770,2855,376.0,135.0,3.0,126321.0,1.0,102391,Parada
2855,2856,1003,1,0.762895,2856,762.0,135.0,5.0,126322.0,1.0,105865,Parada
2856,2857,1003,1,1.523722,2857,1523.0,135.0,16.0,77366.0,1.0,51118,Parada
2857,2858,1003,1,1.723795,2858,1723.0,135.0,20.0,77078.0,-1.0,51122,Parada
2858,2859,1003,1,1.938552,2859,1938.0,135.0,25.0,77082.0,1.0,51126,Parada
2859,2860,1003,1,2.115774,2860,2115.0,135.0,27.0,77080.0,1.0,51125,Parada
2860,2861,1003,1,2.371477,2861,2371.0,135.0,29.0,88846.0,1.0,105862,Parada
2861,2862,1003,1,2.788811,2862,2788.0,135.0,31.0,88847.0,1.0,9084,Parada
2862,2863,1003,1,3.125591,2863,3125.0,135.0,34.0,86886.0,1.0,56959,Parada


In [25]:
sp2024_withSequence[sp2024_withSequence['ROUTE_ID'] == 1004]

,ID,ROUTE_ID,PASS_COUNT,MILEPOST,STOP_ID,USERID,TRACK,SECUENCIA,LINK_ID,DIR,NODE,TIPO
2918,2919,1004,1,0.190977,2919,190.0,136.0,1.0,145013.0,-1.0,105863,Parada
2919,2920,1004,1,0.376770,2920,376.0,136.0,3.0,126321.0,1.0,102391,Parada
2920,2921,1004,1,0.762895,2921,762.0,136.0,5.0,126322.0,1.0,105865,Parada
2921,2922,1004,1,1.523722,2922,1523.0,136.0,16.0,77366.0,1.0,51118,Parada
2922,2923,1004,1,1.723795,2923,1723.0,136.0,20.0,77078.0,-1.0,51122,Parada
2923,2924,1004,1,1.938552,2924,1938.0,136.0,25.0,77082.0,1.0,51126,Parada
2924,2925,1004,1,2.115774,2925,2115.0,136.0,27.0,77080.0,1.0,51125,Parada
2925,2926,1004,1,2.371477,2926,2371.0,136.0,29.0,88846.0,1.0,105862,Parada
2926,2927,1004,1,2.788811,2927,2788.0,136.0,31.0,88847.0,1.0,9084,Parada
2927,2928,1004,1,3.125591,2928,3125.0,136.0,34.0,86886.0,1.0,56959,Parada


### Cantidad de rutas originales(incluyendo las que tienen todas sus paradas SIN SECUENCIA) y rutas finales donde sus paradas SI TIENEN SECUENCIA

In [78]:
#42 rutas tenian todas sus paradas SIN SECUENCIA, Y NO ERAN USABLES
print(f'Total routes in ro2024_df: {ro2024_df["ROUTE_ID"].nunique()}')
print(f'Total routes with SEQUENCE on their stop points: {stop_points_cleaned["ROUTE_ID"].nunique()}')

Total routes in ro2024_df: 796
Total routes with SEQUENCE on their stop points: 754


## Create LINE ROUTES

### __Define search parameters__

#### __HowToHandleIncompleteRoute__
* RouteSearchHandleIncompleteRouteTIgnoreLine = 0
* RouteSearchHandleIncompleteRouteTInsertLink = 1
* RouteSearchHandleIncompleteRouteTOpenLink = 4
* RouteSearchHandleIncompleteRouteTSearchShortestPath = 2 __(uses this)__
* RouteSearchHandleIncompleteRouteTSearchShortestPath = 3


#### __ShortestPathCriterion__
* ShortestPathCriterion_LinkLength = 3 Link Length
* ShortestPathCriterion_LinkLengthDir = 0 Link Direct Distance __(changed it to this)__
* ShortestPathCriterion_LinkT_Sys = 1 Link travel time of current transport system __(this as default but i dont have travel times yet)__
* ShortestPathCriterion_LinktypeT_Sys = 2 Link type travel time of current transport system

#### __WhatToDoIfShortestPathNotFound__
* IfNotFound_DontRead = 0 Do not read __(this as default)__
* IfNotFound_InsertLink = 2 Insert link if necessary
* IfNotFound_OpenLink = 1 Open link or turn for transport system __(changed it to this)__

In [79]:
#Route search params
route_search_params = Visum.IO.CreateNetReadRouteSearchTSys()
route_search_params.SetAttValue("HowToHandleIncompleteRoute", C.RouteSearchHandleIncompleteRouteTSearchShortestPath) #2 RouteSearchHandleIncompleteRouteTSearchShortestPath
route_search_params.SetAttValue("ShortestPathCriterion", C.ShortestPathCriterion_LinkT_Sys) #1 ShortestPathCriterion_LinkT_Sys = link travel time of current transport system
route_search_params.SetAttValue("IncludeBlockedLinks", False)
route_search_params.SetAttValue("IncludeBlockedTurns", False)
route_search_params.SetAttValue("MaxDeviationFactor", 1000)
route_search_params.SetAttValue("WhatToDoIfShortestPathNotFound", 0)

In [91]:
ro2024_df

,ROUTE_ID,ROUTE_NAME,ID_RUTA,TIME,LONGITUD,DISTANCE,RUTA_ACT,RUTA_ANT,CLASE,ORI_DES,...,FLOTA,CAPACIDAD,HEADWAY,TDR,MODE,CORREDOR,DEMANDA1,DEMANDA2,DEMANDA3,F_CORREDOR
0,1,Route 1,1,36.05,9.55,0.0,C33 Via Corta,NaN,Complementarias,NaN,...,0.0,240.0,20.0,31.63,1,NaN,0.0,0.0,0.00,0.0000
1,2,Route 10,10,68.46,16.93,0.0,C02,45A_2,Complementarias,Destino:�Centro Metropolitano,...,17.0,536.0,9.0,60.07,1,NaN,0.0,0.0,0.00,0.0000
2,3,Route 100,100,116.25,39.28,3874.0,C125 Valle de los Emperadores,186 Valle de los Emperadores_2,Complementarias,Destino: Antigua Central de Autobuses,...,29.0,404.2,14.0,121.80,1,11.0,0.0,0.0,2.07,98.6186
3,4,Route 101,101,114.35,38.50,3904.0,C125 Valle de los Emperadores directa,186 Valle de los Emperadores - Direct_1,Complementarias,Origen: Valle de los Emperadores,...,21.0,297.0,18.0,120.11,1,11.0,0.0,0.0,3.16,101.4041
4,5,Route 102,102,116.25,39.28,3874.0,C125 Valle de los Emperadores directa,186 Valle de los Emperadores - Direct_2,Complementarias,Destino: Antigua Central de Autobuses,...,21.0,297.0,18.0,121.80,1,11.0,0.0,0.0,2.07,98.6186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
791,1333,Route 952,952,69.92,17.35,0.0,L3 Alimentadora Milpillas Op3,L3,SITREN,Milpillas,...,5.0,112.0,20.0,81.21,6,10.0,0.0,0.0,0.00,0.0000
792,1334,Route 953,953,94.06,12.54,0.0,L3 Alimentadora Angeles Op3,L3,SITREN,�ngeles,...,9.0,152.0,20.0,86.09,6,NaN,0.0,0.0,0.00,0.0000
793,1335,Route 954,954,98.18,13.31,0.0,L3 Alimentadora Angeles Op3,L3,SITREN,�ngeles,...,9.0,152.0,20.0,89.98,6,NaN,0.0,0.0,0.00,0.0000
794,1336,Route 955,955,83.77,9.64,0.0,L3 Alimentadora Las Agujas Op3,L3,SITREN,T16B-C03,...,20.0,440.0,12.0,76.86,6,NaN,0.0,0.0,0.00,0.0000


In [81]:
ro2024_df = ro2024_df.sort_values(by='ROUTE_ID')


In [56]:
route_stop_points = stop_points_cleaned[stop_points_cleaned['ROUTE_ID'] == 175]
route_stop_points

,ID,ROUTE_ID,PASS_COUNT,MILEPOST,STOP_ID,USERID,TRACK,SECUENCIA,LINK_ID,DIR,NODE,TIPO,StopPointNo,NodeNo
10795,10915,175,1,0.306602,10915,306.0,256.0,4,46153.0,1,32861,Parada,3921,32861
10796,10916,175,1,0.514242,10916,514.0,256.0,6,46416.0,1,32828,Parada,3922,32828
10797,10917,175,1,0.867302,10917,867.0,256.0,12,46046.0,1,32847,Parada,3923,32847
10798,10918,175,1,1.154116,10918,1154.0,256.0,16,46042.0,1,32841,Parada,3924,32841
10799,10919,175,1,1.447206,10919,1447.0,256.0,17,46041.0,1,32843,Parada,3925,32843
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10855,10975,175,1,16.696772,10975,16696.0,256.0,184,35747.0,-1,26532,Parada,517,26532
10856,10976,175,1,17.051222,10976,17051.0,256.0,187,35569.0,-1,26535,Parada,502,26535
10857,10977,175,1,17.320377,10977,17320.0,256.0,190,35837.0,-1,26514,Parada,3947,26514
10858,10978,175,1,17.513462,10978,17513.0,256.0,193,35840.0,-1,26497,Parada,1315,26497


In [82]:
line_no = 1

for row in ro2024_df.itertuples():
    route_id = row.ROUTE_ID
    route_name_ID = row.ID_RUTA

    #Check if route_id exists in stop_points_cleaned ROUTE_ID column, if not skip to next iteration
    if(stop_points_cleaned.loc[stop_points_cleaned['ROUTE_ID'] == route_id].empty):
        print(f"Route ID {route_id} not found in stop points dataframe.")
        continue

     #Create line
    tsys_bus = Net.TSystems.ItemByKey("B")
    line = Net.AddLine('Line ' + str(route_id), tsys_bus)

    #Direction
    direction = Visum.Net.Directions.GetAll[0]

    #Get stop points for route_id
    route_stop_points = stop_points_cleaned[stop_points_cleaned['ROUTE_ID'] == route_id]

    #Sort by SECUENCIA column to ensure correct order of stop points in line route
    route_stop_points = route_stop_points.sort_values(by='SECUENCIA')

    #Create collection of stop points for line route
    route_elements = Visum.CreateNetElements()

    #Add stop points to route elements collection
    for row in route_stop_points.itertuples():
        stop_point_no = row.StopPointNo
        sp = Net.StopPoints.ItemByKey(stop_point_no)
        route_elements.Add(sp)

    #Create line route
    print('creating line route with ROUTE_ID', route_id)
    try:
        line_route = Net.AddLineRoute('LineRoute ' + str(route_id), line, direction, route_elements, route_search_params)
        if line_route is None:
            print(f"⚠️ No se pudo crear LineRoute para Route ID {route_id}: ruta no encontrada, ignorando.")
            continue
        print(f"✅ LineRoute creada para Route ID {route_id}")
    except Exception as e:
        print(f"⚠️ Error al crear LineRoute para Route ID {route_id}: {e}, ignorando.")
        continue

creating line route with ROUTE_ID 1
✅ LineRoute creada para Route ID 1
creating line route with ROUTE_ID 2
✅ LineRoute creada para Route ID 2
creating line route with ROUTE_ID 3
✅ LineRoute creada para Route ID 3
creating line route with ROUTE_ID 4
✅ LineRoute creada para Route ID 4
creating line route with ROUTE_ID 5
✅ LineRoute creada para Route ID 5
creating line route with ROUTE_ID 10
✅ LineRoute creada para Route ID 10
creating line route with ROUTE_ID 11
✅ LineRoute creada para Route ID 11
creating line route with ROUTE_ID 13
✅ LineRoute creada para Route ID 13
creating line route with ROUTE_ID 15
✅ LineRoute creada para Route ID 15
creating line route with ROUTE_ID 16
✅ LineRoute creada para Route ID 16
creating line route with ROUTE_ID 17
✅ LineRoute creada para Route ID 17
creating line route with ROUTE_ID 18
✅ LineRoute creada para Route ID 18
creating line route with ROUTE_ID 19
✅ LineRoute creada para Route ID 19
creating line route with ROUTE_ID 20
✅ LineRoute creada para 

### __comment__ cuando searchparameters usa LinkTsys las LineRoutes salen super extranas (y 4 no se pudieron generar no se por que). 

### __intentar mapear la LineRoute 1 solo para ver si si o no

In [86]:
sp_route1 = stop_points_cleaned[stop_points_cleaned['ROUTE_ID'] == 1]
tsys_bus = Net.TSystems.ItemByKey("B")
line1_test = Net.AddLine('Line1 Test', tsys_bus)

direction = Visum.Net.Directions.GetAll[0]

route_elements = Visum.CreateNetElements()
for row in sp_route1.itertuples():
    stop_point_no = row.StopPointNo
    sp = Net.StopPoints.ItemByKey(stop_point_no)
    route_elements.Add(sp)

search_params_linkTSys = Visum.IO.CreateNetReadRouteSearchTSys()
search_params_linkTSys.SetAttValue("HowToHandleIncompleteRoute", C.RouteSearchHandleIncompleteRouteTSearchShortestPath) #2 RouteSearchHandleIncompleteRouteTSearchShortestPath
search_params_linkTSys.SetAttValue("ShortestPathCriterion", 1) #1 ShortestPathCriterion_LinkT_Sys = link travel time of current transport system
search_params_linkTSys.SetAttValue("IncludeBlockedLinks", False)
search_params_linkTSys.SetAttValue("IncludeBlockedTurns", False)
search_params_linkTSys.SetAttValue("MaxDeviationFactor", 1000)
search_params_linkTSys.SetAttValue("WhatToDoIfShortestPathNotFound", 0)
line_route = Net.AddLineRoute('LineRoute1 TEST', line1_test, direction, route_elements, search_params_linkTSys)



In [106]:
sp2024_withSequence[sp2024_withSequence['ROUTE_ID'] == 3]

,ID,ROUTE_ID,PASS_COUNT,MILEPOST,STOP_ID,USERID,TRACK,SECUENCIA,LINK_ID,DIR,NODE,TIPO
96,97,3,1,0.000000,97,0.0,100.0,1.0,57056.0,-1.0,38741,Parada
97,98,3,1,0.222721,98,222.0,100.0,2.0,145218.0,-1.0,106046,Parada
98,99,3,1,0.630473,99,630.0,100.0,4.0,57058.0,-1.0,26527,Parada
99,100,3,1,0.828130,100,828.0,100.0,6.0,88502.0,1.0,26472,Parada
100,101,3,1,1.038427,101,1038.0,100.0,8.0,35506.0,-1.0,26473,Parada
...,...,...,...,...,...,...,...,...,...,...,...,...
184,185,3,1,38.181557,185,38181.0,100.0,287.0,115114.0,1.0,80243,Parada
185,186,3,1,38.365906,186,38365.0,100.0,288.0,115115.0,1.0,80246,Parada
186,187,3,1,38.606274,187,38606.0,100.0,289.0,115116.0,1.0,80244,Parada
187,188,3,1,39.034855,188,39034.0,100.0,290.0,4613.0,1.0,4114,Parada


In [107]:
no_sequence

,ID,ROUTE_ID,PASS_COUNT,MILEPOST,STOP_ID,USERID,TRACK,SECUENCIA,LINK_ID,DIR,NODE,TIPO
41396,44753,3,1,35.385487,44753,NaN,NaN,NaN,NaN,NaN,4547,Extra
41395,44754,5,1,35.384651,44754,NaN,NaN,NaN,NaN,NaN,4547,Extra
41334,44759,10,1,7.386519,44759,NaN,NaN,NaN,NaN,NaN,98295,Extra
41333,44760,11,1,28.117382,44760,NaN,NaN,NaN,NaN,NaN,98295,Extra
44531,45387,215,1,3.226703,45387,NaN,NaN,NaN,NaN,NaN,72865,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
45230,46207,1333,1,0.032434,46207,NaN,NaN,NaN,NaN,NaN,83646,NaN
45286,46263,1335,1,0.034565,46263,NaN,NaN,NaN,NaN,NaN,83646,NaN
45287,46264,1335,1,0.570972,46264,NaN,NaN,NaN,NaN,NaN,83429,NaN
45313,46290,1336,1,9.784491,46290,NaN,NaN,NaN,NaN,NaN,83653,NaN


In [115]:
#get route_ids where some sp has no freq. but the rest of the route has sp
mostly_complete_routes = no_sequence[no_sequence['ROUTE_ID'].isin(sp2024_withSequence['ROUTE_ID'])]
mostly_complete_routesIDS = mostly_complete_routes['ROUTE_ID'].nunique()
usable_routes = ro2024_df[ro2024_df['ROUTE_ID'].isin(mostly_complete_routes['ROUTE_ID'])]

print(f'Unique values en SITUACION de rutas con algun SP sin SECUENCIA: {usable_routes['SITUACION'].unique()}')
print(f'Unique values en CLASE de rutas con algun SP sin SECUENCIA: {usable_routes['CLASE'].unique()}')
print(f'Total de rutas que tiene algun SP sin SECUENCIA pero aun se pueden formar: {len(usable_routes)}')
usable_routes.head()




Unique values en SITUACION de rutas con algun SP sin SECUENCIA: ['Operando' 'No opera' 'Operando*' 'Solo campo' 'Propuesta']
Unique values en CLASE de rutas con algun SP sin SECUENCIA: ['Complementarias' 'Alimentadora Macrob�s' 'Complementaria de troncal'
 'Troncal' 'Tren ligero' 'BRT' 'Alimentadora Mi Macro Perif�rico'
 'Complementaria Mi Macro Perif�rico' 'SITREN']
Total de rutas que tiene algun SP sin SECUENCIA pero aun se pueden formar: 149


,ROUTE_ID,ROUTE_NAME,ID_RUTA,TIME,LONGITUD,DISTANCE,RUTA_ACT,RUTA_ANT,CLASE,ORI_DES,...,FLOTA,CAPACIDAD,HEADWAY,TDR,MODE,CORREDOR,DEMANDA1,DEMANDA2,DEMANDA3,F_CORREDOR
2,3,Route 100,100,116.25,39.28,3874.0,C125 Valle de los Emperadores,186 Valle de los Emperadores_2,Complementarias,Destino: Antigua Central de Autobuses,...,29.0,404.2,14.0,121.80,1,11.0,0.0,0.0,2.07,98.6186
4,5,Route 102,102,116.25,39.28,3874.0,C125 Valle de los Emperadores directa,186 Valle de los Emperadores - Direct_2,Complementarias,Destino: Antigua Central de Autobuses,...,21.0,297.0,18.0,121.80,1,11.0,0.0,0.0,2.07,98.6186
5,10,Route 107,107,119.10,35.73,2380.0,C127A (No Opera),183A Chulavista_1,Complementarias,Origen:�Villa Fontana Aqua Etapa 6,...,9.0,120.0,40.0,116.82,1,NaN,0.0,0.0,0.00,66.6106
6,11,Route 108,108,118.14,35.50,2055.0,C127A (No Opera),183A Chulavista_2,Complementarias,Destino: Rinconada del Bosque,...,9.0,120.0,40.0,115.86,1,NaN,0.0,0.0,0.00,57.8805
161,215,Route 292,292,87.73,23.64,0.0,C89,162 Copala_1,Complementarias,C89,...,1.0,24.0,180.0,80.25,1,10.0,0.0,0.0,0.00,0.0000


In [117]:
usable_routes.to_csv('Routes_WhereOneSP_HasNoSeq.csv', index=False)

In [116]:
#get route_ids from no_sequence that are not in sp_withSequence
#if the route_id is in sp_withSequence then it's just a stop point in the route that has no seq. & route is mostly complete and therefore usable
#if the route_id is not in sp_withSequence every SP in that route has no seq & route is NOT USABLE AT ALL

#routes where every sp has no seq
unusable_routes = no_sequence[~no_sequence['ROUTE_ID'].isin(sp2024_withSequence['ROUTE_ID'])]

#unique route ids from unusable routes
unusable_routesIDS = unusable_routes['ROUTE_ID'].nunique()
routes_withoutAnySeq = ro2024_df[ro2024_df['ROUTE_ID'].isin(unusable_routes['ROUTE_ID'])]

print(f'Unique values en SITUACION de rutas con todos los SP sin SECUENCIA: {routes_withoutAnySeq['SITUACION'].unique()}')
print(f'Unique values en CLASE de rutas con todos los SP sin SECUENCIA: {routes_withoutAnySeq['CLASE'].unique()}')
print(f'Total de rutas que no tienen SECUENCIA en NI UN SP y no se pueden crear: {len(routes_withoutAnySeq)}')
routes_withoutAnySeq.head()

Unique values en SITUACION de rutas con todos los SP sin SECUENCIA: ['Construcci�n' 'Propuesta' 'Federal']
Unique values en CLASE de rutas con todos los SP sin SECUENCIA: ['Tren ligero' 'SITREN' 'Federal suburbana']
Total de rutas que no tienen SECUENCIA en NI UN SP y no se pueden crear: 42


,ROUTE_ID,ROUTE_NAME,ID_RUTA,TIME,LONGITUD,DISTANCE,RUTA_ACT,RUTA_ANT,CLASE,ORI_DES,...,FLOTA,CAPACIDAD,HEADWAY,TDR,MODE,CORREDOR,DEMANDA1,DEMANDA2,DEMANDA3,F_CORREDOR
647,1104,Route 901,901,NaN,20.48,0.0,L4,NaN,Tren ligero,Tlajomlulco Centro Fray Angelico,...,0.0,6000.0,5.0,40.96,5,NaN,0.0,0.0,0.0,0.0
648,1105,Route 902,902,NaN,20.48,0.0,L4,NaN,Tren ligero,Fray Angelico Tlajomulco Centro,...,0.0,6000.0,5.0,40.97,5,NaN,0.0,0.0,0.0,0.0
649,1108,Route 903,903,NaN,11.08,0.0,SITREN L5,NaN,SITREN,San Agust�n L4,...,0.0,80.0,12.0,30.11,6,NaN,0.0,0.0,0.0,0.0
650,1109,Route 904,904,NaN,11.20,0.0,SITREN L5,NaN,SITREN,San Agust�n L4,...,0.0,80.0,12.0,30.11,6,NaN,0.0,0.0,0.0,0.0
651,1110,Route 905,905,NaN,10.75,0.0,SITREN L6,NaN,SITREN,C�ntaros L4,...,0.0,80.0,12.0,30.11,6,NaN,0.0,0.0,0.0,0.0


In [102]:
routes_withoutAnySeq.to_csv('Routes_WhereEverySP_HasNoFreq.csv', index=False)